# 05 - SQL Analysis
## Greater London House Price Analysis

Este notebook documenta el proceso de petición de consultas del dataset **UK Price Paid** 
en su útlima capa denominada como gold lista para el consumo por parte de los usuarios de negocio. 

### Objetivo
Simular las consultas realizadas por los usuarios de negocio, donde obtienen el valorr final en las tablas creadas.

### Tabla de entrada
`workspace.default.london_gold` — 2,384,979 filas · 22 columnas

### Pipeline
```
Bronze (raw) → Silver (limpia) → Silver enriquecida (nulos tratados) --> Transformaciones (nuevas variables/KPI´s) --> Capa Gold 
--> SQL Queries --> Valor/Conocimiento disponible para negocio
```

## 1. Visualización de los datos de la capa gold

Visualizamos las primeras cinco líneas de esta capa silver, para confirmar sus dimensiones y variables. Seguidamente realizaremos distintas consultas y gráficos sobre las variables disponibles en la capa gold, para facilitar su comprensión y nos permita obtener valor sobre los datos.

In [0]:
sql = spark.table("workspace.default.london_gold")
display(sql.limit(5))

#### A.) Evolución del Mercado (Tendencia Anual)

- El análisis de tendencia anual revela un crecimiento sostenido del precio medio en Greater London, pasando de £290,000 en 2005 a un pico de £711,000 en 2022.

- Impacto en Volumen: Se observa una contracción drástica en la liquidez del mercado. Mientras que en 2006 se registraron 173,000 transacciones, en 2025 el volumen bajó a 81.000.

- Ajuste Reciente: Los datos de 2025 muestran una corrección del precio medio hacia los £655,000, pudiendo reflejar posiblemente el impacto de los tipos de interés elevados en la capacidad de financiación.

In [0]:
%sql
SELECT 
    year, 
    avg_price_annual AS precio_medio_londres,
    COUNT(transaction_id) AS volumen_ventas
FROM workspace.default.london_gold
GROUP BY year, avg_price_annual
ORDER BY year DESC;

Databricks visualization. Run in Databricks to view.

#### B.) Segmentación por Tipo de Propiedad (Últimos 5 años)
No todos los inmuebles se comportan igual. El análisis de los últimos 5 años muestra una jerarquía de precios muy marcada:

- Detached (Independientes): Es el segmento más premium y volátil, alcanzando los £1.28M en 2022, pero ajustándose a £1.14M en 2025.

- Flats (Pisos/Apartamentos): Representan la base del mercado con una media de £510k - £580k. Es el segmento que más ha sufrido la presión de la inflación, manteniendo precios más planos en comparación con las casas unifamiliares.

In [0]:
%sql
SELECT 
    year, 
    property_type_desc, 
    ROUND(AVG(price), 0) AS precio_medio
FROM workspace.default.london_gold
WHERE year >= 2020
GROUP BY year, property_type_desc
ORDER BY year DESC, precio_medio DESC;

Databricks visualization. Run in Databricks to view.

#### C.) Top 10 Distritos más Caros (Media Histórica)
la consulta de los 10 distritos más caros confirma la enorme brecha territorial en la capital:

- Kensington and Chelsea lidera con una media histórica de £1.45M, seguido por City of Westminster (£1.16M).

- Distritos como Hackney han entrado en el Top 10 histórico, lo que sugiere procesos de gentrificación profunda durante las últimas dos décadas.

In [0]:
%sql
SELECT 
    district, 
    ROUND(AVG(price), 0) AS precio_medio_historico,
    COUNT(transaction_id) AS total_transacciones
FROM workspace.default.london_gold
GROUP BY district
ORDER BY precio_medio_historico DESC
LIMIT 10;

Databricks visualization. Run in Databricks to view.

#### D.) Brecha de Precio: Obra Nueva vs. Segunda Mano
Existe una dinámica curiosa en la brecha de precios:

- En 2023, la obra nueva se vendió significativamente más cara (£818k) que la propiedad establecida (£700k), una diferencia de casi el 17%.

- Sin embargo, históricamente (periodo 2009-2013), las propiedades establecidas solían tener precios medios superiores. Esto indica un cambio de tendencia hacia desarrollos de obra nueva enfocados en el segmento de lujo o high-end en la última década.

In [0]:
%sql
SELECT 
    year, 
    property_status, 
    ROUND(AVG(price), 0) AS precio_medio
FROM workspace.default.london_gold
GROUP BY year, property_status
ORDER BY year DESC, property_status;

Databricks visualization. Run in Databricks to view.

#### E.) Detector de "Oportunidades/Outliers" (Propiedades bajo la media)
Esta consulta de "Oportunidades" detectó propiedades vendidas con desviaciones del -99% respecto a la media de su distrito.
- Podrían representar autenticas gangas o lo más probable, ser transacciones entre familiares que no representan el valor real de mercado.

In [0]:
%sql
SELECT 
    transaction_id,
    date_of_transfer,
    district,
    postcode,
    price AS precio_venta,
    avg_price_by_district AS media_distrito_ese_anio,
    price_vs_district_pct AS porcentaje_desviacion
FROM workspace.default.london_gold
WHERE year = 2024 
  AND price_vs_district_pct < -25  -- Un 25% más barato que la media del barrio
ORDER BY price_vs_district_pct ASC
LIMIT 20;

Databricks visualization. Run in Databricks to view.